# 17-bridge CPW–meander · HFSS Lumped RLC Eigenmode

Use the launcher variant to observe modes with two explicit linear
resistive loads. The same nonmetal supports are reused by the [Lumped
Terminal notebook](aedt_hfss_driven_terminal_lumped.ipynb); geometry
does not irrevocably select a solver treatment. Read the [Lumped RLC
requirement](../../../simulation/requirements/lumped-rlc-eigenmode.qmd)
for the system and expected outputs.

This clean source targets [SCGSim v2
RC1](https://github.com/OrPenStrike/scgsim/tree/85d8bedb733b1e17737d935029618351b30ea5de).
Use a separately selected authoring runtime with the PDK candidate, AEDT
2024.2 and PyAEDT 1.3.0 for eventual execution. Rendering runs no code.
No result from an earlier runtime is attached to this source.

## Action and identity

In [ ]:
import json
import subprocess
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

from IPython.display import display
from scgsim.aedt import AedtResources, EigenmodeRunControl, EigenmodeSim, LumpedRlc, resolve_results
from scgsim.geometry import (
    GeometryBuildInput,
    LayoutPolygonSpec,
    LumpedSupport,
    SemanticEntitySpec,
    VacuumRegionSpec,
    apply_vacuum_region_to_stack,
)

import orpen_sc_pdk
from orpen_sc_pdk.cells import cpw_meander_airbridge_lumped_coupon
from orpen_sc_pdk.cells.simulation import get_cpw_meander_airbridge_source_records

orpen_sc_pdk.activate()
WORKFLOW_ACTION = "author"  # author | prepare_handoff | run | analyze
VARIANT = "lumped"
RUN_ID = f"meander-rlc-{uuid4().hex[:12]}"
OUTPUT_ROOT = Path("notebooks/.artifacts/ComponentSimulation/CpwMeanderAirbridge")
RUN_DIR = OUTPUT_ROOT / RUN_ID
RETURNED_RUN_DIR = OUTPUT_ROOT / "returned-run"  # Explicit relocated completed/partial input.
ANALYSIS_KIND = "strict"  # strict | receipt_only (for incomplete returns)
display({name: version(name) for name in ("scgsim", "gdsfactory", "orpen-sc-pdk")})

## Layout and physical stack

<figure>
<img
src="https://raw.githubusercontent.com/OrPenStrike/orpen-sc-pdk/f4f509843c2c7177773798bf6a8a906f94c63a11/docs/_static/images/test-components/driven-terminal-lumped-resonator.png"
alt="Selected launcher source geometry; not a native model." />
<figcaption aria-hidden="true">Selected launcher source geometry; not a
native model.</figcaption>
</figure>

The source is Al M1 0…0.1 µm, Al posts 0.1…3.1 µm and Al decks 3.1…3.4
µm on the PDK Si substrate. The 17 full-landing bridges join Ground
across slots, not Signal. The two launcher locators are on 202/1 at
source Z=0.05 µm; their normals and directed Signal→Ground contacts are
distinct facts.

In [ ]:
if WORKFLOW_ACTION != "analyze":
    coupon = cpw_meander_airbridge_lumped_coupon()
    coupon.plot()

## Source records → caller-owned model

This one shared authoring cell wires the producer’s complete placed
rings to SCGSim’s public input records. It does not fit geometry, infer
connectivity or choose native object names. The source record’s `::`
keys and qualified occurrence/local Entity IDs are explicitly paired
below. Placed body rings are not transformed again; local neutral
supports carry their one recorded affine.

In [ ]:
if WORKFLOW_ACTION != "analyze":
    source = get_cpw_meander_airbridge_source_records(VARIANT)
    # Geometry classification is separate from physical superconductivity.
    # Preserve the full source record, including is_superconducting.
    geometry_material_kinds = {
        "superconductor": "conductor",
        "conductive": "conductor",
        "conductor": "conductor",
        "dielectric": "dielectric",
        "vacuum": "vacuum",
    }
    entity_ids = {
        e["qualified_entity_id"]: f"{e['source_occurrence_path']}/{e['local_semantic_id']}"
        for e in source["entities"]
    }
    # This notebook explicitly adopts the producer's case grouping proposal.
    # It is not PDK-owned or a connectivity result from the native backend.
    final_nets = {
        entity_ids[source_id]: net
        for group, net in (("signal", "Signal"), ("ground", "Ground"))
        for source_id in source["case_net_groups"][group]
    }
    polygons, entities, conductor_layers, solution_regions = [], [], [], {}
    bounds = dict(
        zip(
            ("x_min_um", "y_min_um", "x_max_um", "y_max_um"),
            source["source_envelope"]["bounds_xy_um"],
            strict=True,
        )
    )
    for e in source["entities"]:
        sid = entity_ids[e["qualified_entity_id"]]
        level = source["source_levels"][e["source_level"]]
        material = source["material_records"][e["source_material"]]
        dielectric = material["material_kind"] == "dielectric"
        low, high = e["source_z_interval_um"]
        geometry = {"z_min_um": low, "z_max_um": high, "thickness_um": level["thickness"]}
        metadata = {
            "source_semantic_id": sid,
            "source_record_id": e["qualified_entity_id"],
            "source_occurrence_path": e["source_occurrence_path"],
            "source_local_entity_id": e["local_semantic_id"],
            "body_membership_id": e["body_membership_id"],
        }
        if "aedt_modeling" in level["info"]:
            metadata["aedt_modeling"] = level["info"]["aedt_modeling"]
        polygon_ids = []
        for p in e["placed_polygons"]:
            polygon_ids.append(p["polygon_id"])
            polygons.append(
                LayoutPolygonSpec(
                    polygon_id=p["polygon_id"],
                    layer="/".join(map(str, p["layer"])),
                    exterior=tuple(map(tuple, p["outer_um"])),
                    holes=tuple(tuple(map(tuple, h)) for h in p["holes_um"]),
                    object_name=sid,
                    net_name=final_nets.get(sid),
                    metadata={"coordinate_frame": p["coordinate_frame"]},
                )
            )
        if len(e["placed_polygons"]) == 1:
            p = e["placed_polygons"][0]
            geometry.update(outer_loop=p["outer_um"], hole_loops=p["holes_um"])
        if dielectric:
            geometry["domain_bounds_um"] = bounds
        record = dict(
            semantic_id=sid,
            role="solution_region" if dielectric else "metal",
            material_id=e["source_material"],
            material_kind=geometry_material_kinds[material["material_kind"]],
            priority=level["mesh_order"],
            geometry_kind="domain" if dielectric else "layout_extrusion",
            geometry=geometry,
            metadata=metadata,
        )
        part_role = None if dielectric else level["info"]["part_role"]
        entities.append(
            SemanticEntitySpec(
                **record,
                polygon_ids=tuple(polygon_ids),
                net_id=final_nets.get(sid),
                part_role=part_role,
                host_void_semantic_id=None if dielectric else "VACUUM_REGION",
            )
        )
        if dielectric:
            solution_regions[sid] = record
        else:
            conductor_layers.append({**record, "part_role": part_role})
    materials = {
        name: {
            **record,
            "kind": geometry_material_kinds[record["material_kind"]],
            "permittivity": record["relative_permittivity"],
        }
        for name, record in source["material_records"].items()
    }
    stack = {
        "materials": materials,
        "layers": conductor_layers,
        "solution_regions": solution_regions,
        "metadata": {},
    }
    source_bottom = min(e["source_z_interval_um"][0] for e in source["entities"])
    source_top = max(e["source_z_interval_um"][1] for e in source["entities"])
    envelope_bottom, envelope_top = source["source_envelope"]["z_interval_um"]
    stack = apply_vacuum_region_to_stack(
        stack,
        VacuumRegionSpec(
            0.0,
            0.0,
            0.0,
            0.0,
            source_bottom - envelope_bottom,
            envelope_top - source_top,
        ),
    )
    vacuum = stack["solution_regions"]["VACUUM_REGION"]
    entities.append(
        SemanticEntitySpec(
            semantic_id="VACUUM_REGION",
            role="solution_region",
            material_id=vacuum["material_id"],
            material_kind="vacuum",
            priority=0,
            geometry_kind="domain",
            geometry=vacuum["geometry"],
            metadata=vacuum["metadata"],
        )
    )
    supports = tuple(LumpedSupport.from_payload(s) for s in source["local_supports"])
    build_input = GeometryBuildInput(
        polygons=tuple(polygons),
        entities=tuple(entities),
        solution_regions=stack["solution_regions"],
        lumped_supports=supports,
        metadata={
            "source_dbu_um": source["source_dbu_um"],
            "caller_final_nets": final_nets,
            "source_coordinate_authority": "placed bodies / local supports",
        },
    )
    display(
        {
            "variant": VARIANT,
            "source_levels": source["source_levels"],
            "source_envelope": source["source_envelope"],
            "entity_count": len(source["entities"]),
            "supports": [s.to_payload() for s in supports],
        }
    )

## Loads, enclosure and adaptive search

Initial controls retain the selected source case: **pure R**, parallel
topology, 50 Ω at each launcher; `None` disables L and C. They are
general linear RLC loads, not junctions and not an automatic EPR
request. `MODEL="thin_film"` uses the declared M1/local-deck/contact
map; selecting Solid preserves the same physical source. The finite XY
enclosure and source air top are retained; the HFSS exterior uses its
closed default boundary, not a radiation condition.

In [ ]:
MODEL = "thin_film"  # solid | thin_film; an explicit model choice, not source rewrite.
resistance_ohm = 50.0
inductance_h = None
capacitance_f = None
run_control = EigenmodeRunControl(
    setup_name="Setup1",
    minimum_frequency_ghz=1.0,
    num_modes=1,
    maximum_passes=2,
    minimum_passes=1,
    minimum_converged_passes=1,
    maximum_delta_frequency_percent=1.0,
    percent_refinement=30.0,
)
resources = AedtResources(cores=4, ram_limit_percent=20)
geometry_workers = 1
if WORKFLOW_ACTION != "analyze":
    sim = EigenmodeSim(modeling=MODEL)
    sim.set_geometry(build_input)
    sim.set_stack(stack)
    sim.set_eigenmode(
        project_name=RUN_ID,
        design_name="MeanderRlcEigenmode",
        run_control=run_control,
        resources=resources,
        geometry_workers=geometry_workers,
    )
    for support in supports:
        sim.add_lumped_rlc(
            LumpedRlc(
                support_id=support.support_id,
                topology="parallel",
                resistance_ohm=resistance_ohm,
                inductance_h=inductance_h,
                capacitance_f=capacitance_f,
            )
        )
    sim.set_output_dir(RUN_DIR)
    display(sim.input_summary())

## Prepare / handoff

Preparation is a separate explicit action. Review its source/effective
Z, support membership, enclosure and enabled load components before any
later native run. The resource request is not a host-wide hard RAM cap
or a reserved license. This publication round does not execute the
following preparation.

In [ ]:
HANDOFF = None
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    HANDOFF = sim.prepare_handoff()
    display(HANDOFF)

## Run: explicit native transaction

In [ ]:
if WORKFLOW_ACTION == "run":
    subprocess.run([str(HANDOFF.script_path)], cwd=HANDOFF.run_dir, check=True)

Do not select `run` during documentation authoring. Future execution
must bind actual ownership, allocation and a fresh output directory;
never reuse or restart a stopped historical job.

## Returned Run: new kernel, no source rebuild

Restart the kernel, choose `analyze`, and set the returned directory
explicitly. Strict resolution checks the actual return. For a known
incomplete transaction, `receipt_only` exposes its recorded state
without pretending it is a resolved physics result. No automatic retry
or catch-and-success fallback is used.

In [ ]:
RESULT = None
if WORKFLOW_ACTION == "analyze":
    if ANALYSIS_KIND == "strict":
        RESULT = resolve_results(RETURNED_RUN_DIR)
    elif ANALYSIS_KIND == "receipt_only":
        display(json.loads((RETURNED_RUN_DIR / "metadata/aedt_run_receipt.json").read_text()))
    else:
        raise ValueError("choose strict or receipt_only analysis")

### Physics and numerical history

In [ ]:
if RESULT is not None:
    display(RESULT.physics_results())
    display(RESULT.simulation_benchmark())
    display(RESULT.show_all_results(show_details=True))

Inspect loaded mode frequencies/fields, enabled loads and original units
separately from adaptive history and resource records. A two-pass limit
is not an adequacy claim; retain warnings, missing data and nonconverged
status. This notebook does not infer Q/T1 or compare aligned reference
planes with the geometrically different Wave variant.